## DashScope API

### 阿里云百炼平台

- [百炼主页](https://www.aliyun.com/product/bailian "点击访问阿里云链接")
- [产品文档](https://bailian.console.aliyun.com/?spm=5176.29597918.J_SEsSjsNv72yRuRFS2VknO.2.261d5c6aq5eDhx&tab=doc#/doc "点击访问阿里云链接")
- [API 参考](https://bailian.console.aliyun.com/?spm=5176.29597918.J_SEsSjsNv72yRuRFS2VknO.2.261d5c6aq5eDhx&tab=api#/api "点击访问阿里云链接")

### 基础配置

- api_key: 传入你自己的 API KEY
    - 在代码中显式展示自己的 API KEY 并不是一个好习惯，容易发生 API KEY 泄露
    - 更推荐的方式是将 API KEY 保存到电脑的环境变量再通过代码从环境变量中读取
    - [环境变量配置方法](https://bailian.console.aliyun.com/cn-beijing?spm=5176.29597918.J_bNSze_09Z5SDm7ZHdpz3Y.1.57517ca0i9gpk9&tab=api#/api/?type=model&url=2803795 "点击访问阿里云链接")
- base_url: 传入对应厂商的 base_url 地址
    - 每个大模型厂商有自己的 base_url 地址，切换厂商（如使用火山引擎）时需要更改
    - 若使用的都是百炼平台（北京）的大模型则无需更改
- model: 传入想要调用的模型名称
    - [百炼平台可用模型列表](https://help.aliyun.com/zh/model-studio/models "点击访问阿里云链接")
    - qwen3.5-plus, qwen3.5-flash
    - qwen3-max, qwen-plus, qwen-flash
    - deepseek-v3.2, deepseek-r1

### 环境配置

pip install openai dashscope

### 多轮对话

- 调用 Qwen3.5-Plus 搭建多轮对话系统

In [1]:
import os
from openai import OpenAI

# 设置 API KEY
api_key = "sk-f9c9d25bb85b47aea36b7fc8cf8d3a6c"  # 使用你自己的 API KEY
# api_key = os.environ.get("DASHSCOPE_API_KEY")
if not api_key:
    raise ValueError("环境变量'DASHSCOPE_API_KEY'未设置")

# 初始化 OpenAI 客户端
client = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",  # 设置阿里云专用端点
    api_key=api_key
)

# 将 messages 发送给模型并返回模型回复文本
def get_response(messages):
    completion = client.chat.completions.create(
        model="qwen3.5-plus",
        messages=messages,
        temperature=0.7,
        max_tokens=512
    )
    return completion.choices[0].message.content

In [2]:
# 初始化一个 messages 列表
# 提示词类型："system"、"user"、"assistent"
messages = [
    {"role": "system", "content": "你是一位《大模型技术原理与商业应用》课程的助教，请深入浅出地回答同学们在LLM领域的提问。"}
]

# 开始对话
user_input = input("用户：")
while user_input.lower() != "quit":
    messages.append({"role": "user", "content": user_input})             # 将用户问题信息添加到 messages 列表中
    assistant_output = get_response(messages)                            # 获取大模型输出
    print(f"用户：{user_input}\n模型: {assistant_output}\n")              # 打印对话内容
    messages.append({"role": "assistant", "content": assistant_output})  # 将大模型的回复信息添加到 messages 列表中
    user_input = input("用户：")

用户： 你是谁？


用户：你是谁？
模型: 你好呀，同学！👋

我是这门《大模型技术原理与商业应用》课程的**助教**。你可以把我当作你在 LLM 学习路上的“向导”和“答疑伙伴”。

我的任务是帮你**深入浅出**地打通从“技术原理”到“商业落地”的任督二脉。具体来说，你可以随时向我请教：

1.  **技术原理**：比如 Transformer 架构到底怎么工作？预训练、SFT、RLHF 有什么区别？显存不够怎么办？
2.  **开发实践**：如何写高质量的 Prompt？怎么搭建一个 Agent？模型微调需要多少数据？
3.  **商业应用**：大模型在哪些场景能真正赚钱？如何评估 ROI？未来的行业趋势是什么？

不管你是刚入门的小白，还是想深入钻研的开发者，有任何问题都可以随时抛给我。咱们一起把大模型这件事儿搞懂、用好！🚀

那么，今天想聊点什么？



用户： 用最简练的语言介绍一下diffusion llm


用户：用最简练的语言介绍一下diffusion llm
模型: 你好！这是当前的前沿探索方向，用最简练的话概括：

1.  **本质**：将图像生成中的**扩散模型**迁移到**文本生成**领域。
2.  **机制**：传统 LLM 是**“接龙”**（自回归，逐个预测），它是**“雕刻”**（从随机噪声迭代去噪成文）。
3.  **优势**：支持**并行生成**，理论推理速度更快。
4.  **现状**：研究热点，但生成质量与可控性暂不及主流 Transformer 架构。

希望能帮你快速建立概念！如有兴趣，我们可以深入聊聊它的去噪过程。



用户： quit


### 流式输出

- 之前我们使用的是非流式输出（模型生成完成后一起返回），但对于开启思考模式的模型，使用非流式输出的超时风险较高，建议使用**流式输出**（模型边生成边返回），且对于 Qwen3 等部分模型，**只支持流式输出**，**必须**设置 `stream=True` 开启流式输出，此时获取模型回复的方式会发生变化
- `include_usage` 设置为 `True` 时，最后一次将返回 Token 用量信息；设置为 `False` 时，将返回空，因此循环时需先判断 `chunk.choices`是否为空，否则报错
- 以下代码中不可以设置 `enable_thinking` 为 `True`，否则会报错，推理模式需要额外的条件判断，详见推理模式代码

In [3]:
completion = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=[
        {"role": "user", "content": "你是谁？"}
    ],
    stream=True,                              # 开启流式输出
    stream_options={"include_usage": False},  # 最后是否返回此次 Token 消耗
    extra_body={"enable_thinking": False}     # 是否开启推理模式
)

full_content = ""
print("- 流式输出内容为：")
for chunk in completion:
    if chunk.choices:
        new_content = chunk.choices[0].delta.content
        full_content += new_content
        print(new_content)
print(f"- 完整内容为：\n{full_content}")

- 流式输出内容为：

你好
！我是 Qwen
3.5，
阿里巴巴最新推出的通
义千问大
语言模型。我
具备强大的语言理解
、逻辑推理、
代码生成及多
模态处理能力
，支持全球 
100 
多种语言的流畅交互
。无论是
解答复杂问题、
创作
内容，还是分析
图表
、编写程序，
我都能为你提供
高效精准的帮助。
有什么
具体需求吗？
 😊

- 完整内容为：
你好！我是 Qwen3.5，阿里巴巴最新推出的通义千问大语言模型。我具备强大的语言理解、逻辑推理、代码生成及多模态处理能力，支持全球 100 多种语言的流畅交互。无论是解答复杂问题、创作内容，还是分析图表、编写程序，我都能为你提供高效精准的帮助。有什么具体需求吗？ 😊


### 推理模式

- 设置 `enable_thinking` 决定是否开启推理模式
- 为防止思考过程过长，可以设置 `thinking_budget` 参数进行截断，即思考消耗 Token 达到指定阈值后将终止思考并开始回复

In [4]:
def print_completion(completion):

    reasoning_content = ""  # 完整思考过程
    answer_content = ""     # 完整回复
    is_answering = False    # 是否进入回复阶段
    
    print("\n" + "=" * 20 + "思考过程" + "=" * 20 + "\n")
    
    for chunk in completion:
        
        # 最后输出此次 Token 消耗
        if not chunk.choices:
            print("\n\n" + "=" * 20 + "Usage" + "=" * 20 + "\n")
            print(f"Usage: {chunk.usage}")
            continue
    
        # 获取当前输出
        delta = chunk.choices[0].delta
    
        # 只收集思考内容
        if hasattr(delta, "reasoning_content") and delta.reasoning_content:
            if not is_answering:
                print(delta.reasoning_content, end="", flush=True)  # flush=True: 强制将缓冲区中的内容立即输出到屏幕，实时输出信息
            reasoning_content += delta.reasoning_content
    
        # 收到 content，开始进行回复
        if hasattr(delta, "content") and delta.content:
            if not is_answering:
                print("\n\n" + "=" * 20 + "完整回复" + "=" * 20 + "\n")
                is_answering = True
            print(delta.content, end="", flush=True)
            answer_content += delta.content

In [5]:
completion = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=[
        {"role": "user", "content": "9.11和9.8谁大？"}
    ],
    stream=True,                             # 开启流式输出
    stream_options={"include_usage": True},  # 最后是否返回此次 Token 消耗
    extra_body={
        "enable_thinking": True,             # 是否开启推理模式
        "thinking_budget": 1024              # 思考 Token 截断阈值
    }
)

print_completion(completion)


====================思考过程====================

Thinking Process:

1.  **Analyze the Request:**
    *   The user is asking "9.11 和 9.8 谁大？" (Which is larger, 9.11 or 9.8?).
    *   This is a common trick question or a test of numerical comparison, often used to probe AI reasoning capabilities regarding decimal numbers.

2.  **Identify the Numbers:**
    *   Number A: 9.11
    *   Number B: 9.8

3.  **Perform Numerical Comparison:**
    *   Compare the integer parts: Both are 9.
    *   Compare the tenths place (first digit after the decimal point):
        *   9.11 has 1.
        *   9.8 has 8.
    *   Since 8 > 1, 9.8 is larger than 9.11.
    *   Alternatively, convert to same decimal places: 9.80 vs 9.11. Clearly 80 > 11.

4.  **Consider Context/Potential Pitfalls:**
    *   Sometimes people confuse version numbers (e.g., software versions) with decimal numbers. In versioning, 9.11 might be considered "newer" or "higher" than 9.8 depending on the scheme, but mathematically, 9.8 > 9.11

### 非推理模式

- 仅修改 `extra_body` 参数
- 建议每次调用时都显示指定 `enable_thinking` 参数

In [6]:
completion = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=[
        {"role": "user", "content": "9.11和9.8谁大？"}
    ],
    stream=True,                             # 开启流式输出
    stream_options={"include_usage": True},  # 最后是否返回此次 Token 消耗
    extra_body={"enable_thinking": False}    # 是否开启推理模式
)

print_completion(completion)


====================思考过程====================



====================完整回复====================

要比较 **9.11** 和 **9.8** 的大小，我们可以从高位到低位依次对比：

1.  **整数部分**：两个数都是 **9**，相等。
2.  **小数部分第一位（十分位）**：
    *   9.11 的十分位是 **1**。
    *   9.8 的十分位是 **8**。

因为 8 大于 1，所以 **9.8 更大**。

为了更直观地对比，也可以将两个数补齐到相同的小数位数（两位小数）：
*   9.11
*   9.80

显然，**9.80 > 9.11**。

结论：**9.8 大**。

====================Usage====================

Usage: CompletionUsage(completion_tokens=169, prompt_tokens=23, total_tokens=192, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=None, rejected_prediction_tokens=None, text_tokens=169), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=None, text_tokens=23))


### 联网搜索

- 仅修改 `messages`、`extra_body` 参数
- 设置 `enable_search` 决定是否开启联网模式，模型将先判断问题是否有必要启动联网搜索，若不必要，则不会进行联网搜索

In [7]:
completion = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=[
        {"role": "user", "content": "飞驰人生3当前票房是多少？"}
    ],
    stream=True,                             # 开启流式输出
    stream_options={"include_usage": True},  # 最后是否返回此次 Token 消耗
    extra_body={
        "enable_thinking": True,             # 是否开启推理模式
        "thinking_budget": 2048,             # 思考 Token 截断阈值
        "enable_search": True                # 是否开启联网模式
    }
)

print_completion(completion)


====================思考过程====================

用户询问《飞驰人生 3》的当前票房。我需要根据知识库中的信息来回答这个问题。

从知识库中，我可以看到多个关于《飞驰人生 3》票房的数据，时间跨度从 2026 年 2 月 17 日到 2026 年 3 月 10 日。当前系统时间是 2026 年 3 月 10 日。

让我整理一下知识库中的票房数据：

1. 2026 年 2 月 17 日：累计票房 3.17 亿
2. 2026 年 2 月 18 日（上映 2 天）：7.55 亿
3. 2026 年 2 月 24 日（上映 8 天）：29.53 亿
4. 2026 年 2 月 26 日（上映 10 天）：33.14 亿
5. 2026 年 3 月 7 日 13 时许：突破 40 亿元
6. 2026 年 3 月 8 日 18 时 48 分：突破 41 亿元
7. 2026 年 3 月 8 日（上映 22 天）：41.28 亿
8. 2026 年 3 月 9 日：累计票房突破 41.11 亿
9. 2026 年 3 月 10 日：有数据显示但票房数字不清晰

最新的明确数据是 2026 年 3 月 8 日的 41.28 亿和 3 月 9 日的 41.11 亿（这个数据似乎有点矛盾，3 月 9 日应该比 3 月 8 日高）。

根据 3 月 8 日的数据，票房已突破 41 亿元，最具体的数字是 41.28 亿。这是目前知识库中最准确和最新的票房信息。

====================完整回复====================

根据知识库中的最新数据，《飞驰人生 3》的当前票房情况如下：

| 时间 | 票房数据 |
|------|----------|
| 2026 年 3 月 8 日 18 时 48 分 | 突破 **41 亿元** |
| 2026 年 3 月 8 日（上映 22 天） | **41.28 亿元** |
| 2026 年 3 月 9 日 | 突破 **41.11 亿元** |

**最新票房：约 41.28 亿元**

此外，《飞驰人生 3》还取得了以下成绩：
- 成为中国影史第 13 部票房 40 亿 + 电影
- 连续 20 天拿下单日票房冠军
- 导演韩寒跻身百亿导演之列

> ⚠️

### 结构化输出

- 适用场景：希望模型返回多类信息，并且后续能便捷地从返回的文本中提取所有信息，而不是将模型返回的文本当作一个整体处理
    - e.g. 信息抽取任务：要求模型从一篇新闻报道中提取出时间、地点、人物、事件
        - 希望模型返回的是：{"时间": "2025-06-01 09:00:00", "地点": "北京市", "人物": ["张三", "李四"], "事件": "张三和李四在北京参加了一个重要会议"}
        - 不希望模型返回一整段文本，对于较为复杂的问题，模型的表述难以捉摸，正则表达式将很难准确提取所需的信息
    - 所谓结构化输出，即**返回一个符合 JSON 格式要求的文本**，便于后续进一步提取
- 方法：
    - 设置请求参数 `response_format` 为 `{"type": "json_object"}`
    - 在提示词中需要指引模型输出 JSON 字符串，否则会报错
    - 建议在提示词中说明每个属性的数据类型，并提供样例给大模型参考

In [8]:
import json

# 预定义示例响应（用于 few-shot 提示）
example1_response = json.dumps(
    {
        "info": {"name": "张三", "age": "25岁", "email": "zhangsan@example.com"},
        "hobby": ["唱歌"]
    },
    ensure_ascii=False  # 不将字符转为 ASCII 码
)
example2_response = json.dumps(
    {
        "info": {"name": "李四", "age": "30岁", "email": "lisi@example.com"},
        "hobby": ["跳舞", "游泳"]
    },
    ensure_ascii=False
)
example3_response = json.dumps(
    {
        "info": {"name": "王五", "age": "40岁", "email": "wangwu@example.com"},
        "hobby": ["Rap", "篮球"]
    },
    ensure_ascii=False
)

completion = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=[
        {
            "role": "system",  # 提示词中需要指引模型输出 JSON 字符串
            "content": f"""提取name、age、email和hobby（数组类型），输出包含info层和hobby数组的JSON。
            示例：
            Q：我叫张三，今年25岁，邮箱是zhangsan@example.com，爱好是唱歌
            A：{example1_response}
            
            Q：我叫李四，今年30岁，邮箱是lisi@example.com，平时喜欢跳舞和游泳
            A：{example2_response}
            
            Q：我的邮箱是wangwu@example.com，今年40岁，名字是王五，会Rap和打篮球
            A：{example3_response}"""
        },
        {
            "role": "user",
            "content": "大家好，我叫赵六，今年34岁，邮箱是liuzhao@example.com，平时喜欢打篮球和旅游", 
        },
    ],
    stream=True,                              # 开启流式输出
    stream_options={"include_usage": False},  # 最后是否返回此次 Token 消耗
    extra_body={"enable_thinking": False},    # 是否开启推理模式
    response_format={"type": "json_object"}   # 指定返回格式
)

# 非流式输出获取回复
# json_string = completion.choices[0].message.content

# 流式输出获取回复
json_string = ""
for chunk in completion:
    if chunk.choices:
        json_string += chunk.choices[0].delta.content

# 解析 JSON 字符串
print(json_string)
json_object = json.loads(json_string)
print(f"json_object['info']['age']: {json_object['info']['age']}")

{"info": {"name": "赵六", "age": "34 岁", "email": "liuzhao@example.com"}, "hobby": ["打篮球", "旅游"]}
json_object['info']['age']: 34 岁


### 图像上传

- Qwen3.5-Plus 为多模态模型，支持图像上传，也可以选择 Qwen3-VL 系列的视觉语言模型
- 若拥有待传图像的 URL 地址，则直接上传即可
- 多图输入仅需增加若干 `{"type": "image_url","image_url": {"url": "xxxxx"}}` 参数即可

In [9]:
completion = client.chat.completions.create(
    model="qwen3.5-plus",  # 可选 qwen3.5-flash、qwen3-vl-plus、qwen3-vl-flash 等
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": "https://scorpio-yyy.oss-cn-shanghai.aliyuncs.com/p2923330987.jpg"},  # 图像 URL 地址
                },
                {"type": "text", "text": "图中是哪一部电影？"},
            ],
        },
    ],
    stream=True,                             # 开启流式输出
    stream_options={"include_usage": True},  # 最后是否返回此次 Token 消耗
    extra_body={"enable_thinking": False}    # 是否开启推理模式
)

print_completion(completion)


====================思考过程====================



====================完整回复====================

图中是电影《长安的荔枝》。

这是一部2025年上映的中国古装喜剧电影，由大鹏执导并领衔主演，改编自马伯庸的同名小说。影片讲述了唐朝小吏李善德（大鹏 饰）被迫承担从岭南运送新鲜荔枝到长安的艰难任务，在各方势力周旋中展开一场充满荒诞与智慧的“荔枝转运”冒险故事。海报上醒目的红色荔枝和众多角色形象也体现了影片的热闹氛围与群像特色。

====================Usage====================

Usage: CompletionUsage(completion_tokens=105, prompt_tokens=115, total_tokens=220, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=None, rejected_prediction_tokens=None, text_tokens=105), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=None, image_tokens=98, text_tokens=17))


- 若是本地图像需要先进行 base64 编码再上传
- 仅修改 `url` 和 `text` 参数

In [10]:
import base64

# 将本地图像转换为 base64 编码
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

base64_image = encode_image("file/rainier.jpg")

completion = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpg;base64,{base64_image}"},  # 按图像格式类型相应修改
                },
                {"type": "text", "text": "描述这张图片"},
            ],
        },
    ],
    stream=True,                             # 开启流式输出
    stream_options={"include_usage": True},  # 最后是否返回此次 Token 消耗
    extra_body={"enable_thinking": False}    # 是否开启推理模式
)

print_completion(completion)


====================思考过程====================



====================完整回复====================

这张图片展现了一幅壮丽的高山山谷全景，充满自然野趣与宁静氛围。

**整体构图：**
画面采用高角度俯瞰视角，前景是几棵挺拔的针叶树（如松树或冷杉），枝叶清晰，为画面增添了层次感和深度。中景是一片开阔的山谷，一条蜿蜒的公路穿过绿意盎然的草地和森林，路边停着多辆汽车，暗示这是一个受欢迎的观景点或旅游区。山谷右侧有一座显眼的木结构建筑，可能是游客中心、 lodge 或餐厅，屋顶呈深棕色，墙体为原木色，与自然环境和谐相融。左侧山谷弥漫着薄雾或晨霭，轻柔地笼罩在树林间，增添了一丝神秘与诗意。

**背景山脉：**
远处是连绵起伏、层峦叠嶂的山脉，山峰轮廓分明，部分山顶仍残留着积雪或裸露岩石，在阳光照射下呈现出灰白、淡紫与青蓝色调。山体被茂密的森林覆盖，颜色从近处的翠绿过渡到远处的蓝绿色，显示出大气透视的效果。光影交错，阳光洒在某些山坡上，形成明亮的暖黄色区域，而另一些则处于阴影之中，呈现深邃的蓝紫色，增强了立体感和戏剧性。

**天空与光线：**
天空占据画面上部约三分之一，云层丰富多变——既有蓬松的白色积云，也有较厚的灰色层云，云朵边缘被阳光染成柔和的粉橙色，表明拍摄时间可能是在清晨或傍晚（黄金时刻）。光线斜射下来，在山体和谷地投下长长的影子，营造出温暖而宁静的氛围。

**色彩与情绪：**
整幅图像色彩饱和度高但不过分艳丽，以绿色、蓝色、棕色为主色调，点缀着阳光的金色与云霞的淡彩。整体情绪宁静、辽阔、壮美，令人感受到大自然的宏伟与人类活动的渺小，也传递出一种远离尘嚣、回归自然的治愈感。

**总结：**
这是一张极具视觉冲击力和艺术美感的风光摄影作品，完美捕捉了高山峡谷在特定光线下的瞬息之美，既展现了自然景观的磅礴气势，又融入了人文元素（道路、车辆、建筑），使画面更具生活气息和故事性。它很可能拍摄于美国落基山脉国家公园（如黄石或大提顿）或其他类似地貌的著名山区景区。

====================Usage====================

Usage: CompletionUsage(completion_tokens=482, prompt_tokens=25

### 文件上传

- Qwen3.5-Plus 支持文本、图像和视频输入，**不支持文件上传**，若需上传文件，有两种方法：
    - 若文件较小，可以先使用 Python 相关库提取文件中的文本，再传给 Qwen3.5
    - 若文件较大，可以选择 Qwen-Long 模型直接上传
    - 以下分别演示两种上传方式

#### 1. 以 PDF 文件为例演示方法一
- 环境配置：pip install PyPDF2

In [11]:
from PyPDF2 import PdfReader

# 提取 PDF 文件中的文本
def extract_text_with_pypdf2(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

# 提取示例
pdf_text = extract_text_with_pypdf2("file/搜索引擎技术.pdf")
print(pdf_text[:1000])

搜索引擎技术
王树森 著2目录
第一部分 搜索引擎基础 1
1搜索引擎技术概要 3
1.1基本概念 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3
1.2什么决定用户满意度？ . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4
1.3搜索引擎链路 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 7
1.4知识点小结 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 10
2搜索引擎的评价指标 11
2.1用户规模与留存指标 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 11
2.2中间过程指标 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 13
2.3人工体验评估 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 15
2.4知识点小结 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 18
第二部分 机器学习基础 21
3机器学习任务 23
3.1二分类任务 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 23
3.2多分类任务 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 24
3.3回归任务 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 26
3.4排序任务 . . . . . . .

- 将文件文本添加到上下文中（仅修改 `messages` 参数）
- 为避免模型混淆角色设定与文档内容，需要在 `messages` 的第一条消息中添加用于角色设定的信息
- 不应将文件文本添加在用于角色扮演的系统提示词和用户提示词中
- 以文本形式传入，总 Token 数受模型上下文长度限制

In [12]:
completion = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=[
        {"role": "system", "content": "你是一名专业的搜索算法工程师"},
        {"role": "system", "content": pdf_text[:100000]},
        {"role": "user", "content": "简要总结pdf中的内容"}
    ],
    stream=True,                             # 开启流式输出
    stream_options={"include_usage": True},  # 最后是否返回此次 Token 消耗
    extra_body={"enable_thinking": False}    # 是否开启推理模式
)

print_completion(completion)


====================思考过程====================



====================完整回复====================

这份文档是王树森所著的《搜索引擎技术》一书的核心内容概要，系统地构建了从理论基础到工业界实战的完整知识体系。全书共分为七个部分，主要涵盖了以下核心内容：

### **第一部分：搜索引擎基础**
*   **核心目标**：提升用户满意度。
*   **五大关键因子**：**相关性**（最重要）、内容质量（E-A-T 原则）、时效性、地域性、个性化。
*   **评价指标**：
    *   **业务指标**：DAU/MAU、留存率（核心北极星指标）。
    *   **中间指标**：点击率 (CTR)、有点比、首点位置、交互率、换词率。
    *   **人工评估**：Side-by-Side (SBS) 对比、DCG/NDCG 评分。
*   **基本链路**：查询词处理 (QP) $\rightarrow$ 召回 (Retrieval) $\rightarrow$ 排序 (Ranking)。

### **第二部分：机器学习基础**
*   **四大任务**：二分类（如 CTR 预估）、多分类（如类目识别）、回归（如相关性打分）、排序（Learning to Rank）。
*   **离线评价指标**：
    *   **Pointwise**：准确率、召回率、F1、AUC（关注单个样本预测准确度）。
    *   **Pairwise**：正逆序比（关注样本对的相对顺序）。
    *   **Listwise**：NDCG（关注整个列表的排序质量，越靠前权重越大）。
*   **NLP 模型训练流程**：预训练 (Pretrain, 如 BERT 的 MLM/SOP) $\rightarrow$ 后预训练 (Post-pretrain, 利用搜索日志挖掘弱监督数据) $\rightarrow$ 微调 (Fine-tune, 人工标注数据) $\rightarrow$ 蒸馏 (Distill, 大模型教小模型以降低线上成本)。

### **第三部分：决定用户体验的关键因子**
*   **相关性**：
    *   定义与分档（高/中/低/无）。
    * 

#### 2. 通过 file-id 传入文档信息（仅能搭配 Qwen-Long 模型使用）

- 将文件通过 OpenAI 兼容接口上传到阿里云百炼平台，保存至平台安全存储空间（免费存储）后获取 file-id

In [13]:
from pathlib import Path

file_object = client.files.create(file=Path("file/搜索引擎技术.pdf"), purpose="file-extract")
print(file_object.id)

file-fe-84dac62d409c4500b02879e0


- 提示词要求同方法一
- 仅修改 `model`、`messages` 参数并删除 `extra_body` 参数
- Qwen Long 上下文长度可达 1000 万 Token（约 1500 万字）
- 若需传入多文档，输入格式为：`f"fileid://{file_object1.id},fileid://{file_object2.id}"`

In [14]:
completion = client.chat.completions.create(
    model="qwen-long",
    messages=[
        {"role": "system", "content": "你是一名专业的搜索算法工程师"},
        {"role": "system", "content": f"fileid://{file_object.id}"},  # 传入 file-id
        {"role": "user", "content": "简要总结pdf中的内容"}
    ],
    stream=True,                             # 开启流式输出
    stream_options={"include_usage": True}   # 最后是否返回此次 Token 消耗
)

print_completion(completion)


====================思考过程====================



====================完整回复====================

这份名为《搜索引擎技术》的PDF文档，系统性地阐述了现代搜索引擎的设计原理、核心技术、评价指标和优化方法。

**核心观点与目标**

文档开篇即明确指出，搜索引擎迭代优化的核心目标是**提升用户满意度**。用户满意度主要由五大维度决定：**相关性、内容质量、时效性、个性化和地域性**。其中，相关性是最重要的基础，其他维度则是在相关性基础上的优化。

**整体架构：搜索链路**

搜索引擎的工作流程被分解为一个清晰的**链路**，包含三个核心环节：

1.  **查询词处理 (Query Processing, QP)**：这是链路的第一环，负责从用户输入的查询词中提取深层信息。主要任务包括：
    *   **分词与词权重**：将查询词切分成词语，并判断每个词的重要性。
    *   **查询词改写**：将原始查询词改写为其他表达方式，以召回更多相关文档。
    *   **意图识别**：识别查询词的时效性、地域性、求购等深层意图，以决定后续链路的调用。

2.  **召回 (Retrieval)**：目标是从海量文档库中快速筛选出数万篇可能相关的候选文档。主要召回通道包括：
    *   **文本召回**：基于倒排索引，通过分词和布尔逻辑（AND/OR）进行精确匹配。
    *   **向量召回**：利用双塔模型将查询词和文档表征为向量，通过向量相似度进行语义召回，解决“语义鸿沟”问题。
    *   **离线召回**：通过离线挖掘（如曝光日志、反向召回）预先计算并存储高相关性的`(q, d)`对，作为高效的补充通道。

3.  **排序 (Ranking)**：这是一个多级漏斗，旨在将召回的候选文档按综合得分排序，最终将最优质的数百篇文档呈现给用户。排序通常分为三级：
    *   **召回海选**：对数万篇文档进行快速初筛。
    *   **粗排**：对数千篇文档进行更精细的打分。
    *   **精排**：对数百篇文档使用最复杂的模型进行最终排序。

**核心技术详解**

*   **评价指标**：文档强调了多层次的评价体系。
    *   *

### 其他补充

- [上下文缓存](https://bailian.console.aliyun.com/cn-beijing/?spm=5176.29597918.J_SEsSjsNv72yRuRFS2VknO.2.261d5c6aq5eDhx&tab=doc#/doc/?type=model&url=2862577 "点击访问阿里云链接")
- [批量推理](https://bailian.console.aliyun.com/cn-beijing/?spm=5176.29597918.J_SEsSjsNv72yRuRFS2VknO.2.261d5c6aq5eDhx&tab=doc#/doc/?type=model&url=2864784 "点击访问阿里云链接")

### 案例演示

- 酒店评论质量评估

In [15]:
import pandas as pd

# 加载数据
file = "file/guangzhou_garden_hotel_comments_2023-2025.csv"
df = pd.read_csv(file, index_col=0)

print(f"共 {len(df)} 条评论")
df.head(1)

共 2607 条评论


,comment,images,score,publish_date,room_type,travel_type,useful_count,review_count
_id,,,,,,,,
68027895e3c98b0941765706,房间非常好 装修很厚重奢华 一开始看评论 看酒店自己po的照片 感觉跟快捷酒店一样 有些害怕...,"[ ""https://dimg04.c-ctrip.com/images/0230y1200...",5.0,2025年4月5日,红棉大床套房,家庭亲子,0,7条点评


In [16]:
from dashscope import Generation

# LLM 客户端
class LLMClient:
    """Qwen 客户端封装"""
    def __init__(self, api_key: str, model: str = "qwen3.5-plus", json: bool = False):
        self.api_key = api_key
        self.model = model
        self.json = json
    
    def generate(self, prompt: str, temperature: float = 0.7) -> str:
        """生成文本"""
        response = Generation.call(
            api_key=self.api_key,
            model=self.model,
            prompt=prompt,
            temperature=temperature,
            result_format="message",
            response_format={"type": "json_object"} if self.json else None
        )
        
        if response.status_code == 200:
            return response.output.choices[0].message.content.strip()
        else:
            raise RuntimeError(f"LLM 调用失败: {response.message}")

In [17]:
# 质量评估 Prompt
QUALITY_ASSESSMENT_PROMPT = """
你是一位专业的评论质量评估专家,请对以下酒店评论进行质量打分(0-10分)

评分标准:
- 9-10分: 信息丰富,具体详细,有明确的观点和充分的细节支持
- 7-8分: 有一定信息量,提供了较为具体的内容
- 5-6分: 信息量中等,有些笼统但仍有参考价值
- 3-4分: 信息量较少,过于简短或模糊
- 0-2分: 几乎没有实质内容,如只有"好"、"不错"等

评论内容: {comment}

直接返回JSON格式:
{{
    "quality_score": 分数(0-10的整数)
}}
"""

def assess_quality(comment: str, index: int) -> int:
    """评估单条评论质量"""
    prompt = QUALITY_ASSESSMENT_PROMPT.format(comment=comment)
    
    for i in range(3):
        try:
            response = llm_client.generate(prompt, temperature=0.3)
            response = response.replace("```json", "").replace("```", "").strip()
            data = json.loads(response)
            return int(data["quality_score"])
        except Exception as e:
            print(f"索引为 {index} 的评论第 {i+1} 次尝试失败: {e}")
            if i < 2:
                time.sleep(1)
                continue

    print(f"索引为 {index} 的评论评估失败，已返回 -1")
    return -1

- 以上提示词是否充分利用了上下文缓存？是否存在改进空间？

In [18]:
# 批量质量评估
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# 客户端初始化
llm_client = LLMClient(api_key=api_key, model="qwen-plus", json=True)

quality_results = {}
with ThreadPoolExecutor(max_workers=16) as executor:
    # 多线程并行
    futures = {executor.submit(assess_quality, row["comment"], index): index for index, row in df.iterrows()}
    for future in tqdm(as_completed(futures), total=len(df), desc="质量评估"):
        index = futures[future]
        quality_results[index] = future.result()

df["quality_score"] = df.index.map(quality_results)

print(f"平均质量分: {df['quality_score'].mean():.2f}")
print(f"质量分布:\n{df['quality_score'].value_counts().sort_index()}")

质量评估:   0%|          | 0/2607 [00:00<?, ?it/s]

平均质量分: 6.74
质量分布:
quality_score
0      10
1      19
2     121
3     105
4     257
5     143
6     254
7     586
8     420
9     643
10     49
Name: count, dtype: int64
